# Lecture 04
## Polynomial regression, locally weighted regression, Maximum likelihood estimation

### A look at the data to check if the linear hypothesis actually fits

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zohaib30/streeteasy-dataset")

CSV_PATH = path+"/manhattan.csv"

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


#CSV_PATH = ""
df = pd.read_csv(CSV_PATH)
X_COL = "size_sqft"
Y_COL = "rent"

N_INTERVALS = 10

# Divide x into n equidistant intervals and compute the mean y per interval.
bins = pd.cut(df[X_COL], bins=N_INTERVALS)
bin_means = df.groupby(bins, observed=True)[Y_COL].mean()
bin_centers = bin_means.index.map(lambda interval: interval.mid)


plt.scatter(df[X_COL], df[Y_COL], alpha=0.4, label="data")
plt.scatter(bin_centers, bin_means.values, color="red", s=60, zorder=5, label=f"mean {Y_COL} per interval")
plt.xlabel(X_COL)
plt.ylabel(Y_COL)
plt.title("Scatterplot with interval means")
plt.legend()
plt.tight_layout()
plt.savefig("/Users/jbenno/temp/plot1.png", dpi=150)

## Polynomial regression

In [ ]:
"""
Polynomial regression for a one-dimensional feature and one-dimensional
target, implemented explicitly and solved in closed form.

Model:  y_hat = w_0 + w_1*x + w_2*x^2 + ... + w_grade*x^grade
Loss:   Mean Squared Error, L(w) = (1/n) * sum((y_hat_i - y_i)^2)

Writing the model in matrix form as y_hat = X @ w, where each row of X
is [1, x_i, x_i^2, ..., x_i^grade] (the Vandermonde matrix), the MSE
becomes

    L(w) = (1/n) * (X @ w - y).T @ (X @ w - y)

Setting the gradient dL/dw to zero gives the normal equations:

    X.T @ X @ w = X.T @ y
    w = (X.T @ X)^-1 @ X.T @ y

which is solved directly below (via np.linalg.solve, not a regression
routine) rather than by gradient descent.

Set the CSV_PATH to the data to analyze (for the rental properties data use the Kaggle script of lecture 03).
Set GRADE to the degree of the polynome you want to fit

Set the input and output variables X_COL and Y_COL.
"""

import csv

import numpy as np
import matplotlib.pyplot as plt

#CSV_PATH = ""

GRADE = 5  # degree of the polynomial

X_COL = "size_sqft"
Y_COL = "rent"

def load_csv(path):
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    x = np.array([float(r[X_COL]) for r in rows])
    y = np.array([float(r[Y_COL]) for r in rows])
    return x, y


def build_design_matrix(x, grade):
    """Build the Vandermonde matrix [1, x, x^2, ..., x^grade] for each x_i."""
    return np.column_stack([x ** power for power in range(grade + 1)])


def fit_polynomial_regression(x, y, grade):
    X = build_design_matrix(x, grade)
    w = np.linalg.solve(X.T @ X, X.T @ y)
    return w


def predict(x, w):
    grade = len(w) - 1
    X = build_design_matrix(x, grade)
    return X @ w


def main():
    x, y = load_csv(CSV_PATH)
    print(f"Loaded {len(x)} points from {CSV_PATH}")

    w = fit_polynomial_regression(x, y, GRADE)
    y_fit = predict(x, w)
    final_mse = np.mean((y_fit - y) ** 2)

    terms = " + ".join(
        f"{coef:.6f}*x^{power}" if power > 0 else f"{coef:.6f}"
        for power, coef in enumerate(w)
    )
    print(f"\nFinal model (grade {GRADE}): y_hat = {terms}")
    print(f"Final MSE: {final_mse:.6f}")

    order = np.argsort(x)
    x_smooth = np.linspace(x.min(), x.max(), 300)
    y_smooth = predict(x_smooth, w)

    plt.figure(figsize=(6.5, 4.5))
    plt.scatter(x, y, alpha=0.6, label="data")
    plt.plot(x_smooth, y_smooth, color="red", linewidth=2,
              label=f"polynomial fit (grade={GRADE})")
    plt.xlabel(X_COL)
    plt.ylabel(Y_COL)
    plt.title(f"Polynomial regression (grade {GRADE})")
    plt.legend()
    plt.tight_layout()

if __name__ == "__main__":
    main()

## Locally weighted linear regression

In [ ]:
"""
Locally weighted regression (LWR) for a one-dimensional feature and
one-dimensional target, implemented explicitly.

Unlike ordinary linear regression, which fits a single global (w, b) to
all data, locally weighted regression fits a separate weighted linear
model for each query point x0. Training points near x0 are given more
weight than points further away, using a Gaussian kernel:

    weight_i(x0) = exp( -(x_i - x0)^2 / (2 * tau^2) )

tau (the bandwidth) controls how "local" the fit is: small tau gives a
wiggly curve that follows the data closely, large tau approaches the
single global linear fit.

For each query point x0, the local (w, b) minimising the weighted MSE

    L(w, b) = sum_i weight_i(x0) * (w*x_i + b - y_i)^2

is obtained the same way as ordinary least squares, but with weighted
means, weighted covariance, and weighted variance in place of their
unweighted counterparts:

    w = sum(weight_i * (x_i - x_mean_w) * (y_i - y_mean_w))
        / sum(weight_i * (x_i - x_mean_w)^2)
    b = y_mean_w - w * x_mean_w

"""

import csv

import numpy as np
import matplotlib.pyplot as plt

#CSV_PATH = ""

TAU = 3          # bandwidth of the Gaussian kernel
N_QUERY_POINTS = 200  # number of points along x at which to evaluate the fit

X_COL = "size_sqft"
Y_COL = "rent"

def load_csv(path):
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    x = np.array([float(r[X_COL]) for r in rows])
    y = np.array([float(r[Y_COL]) for r in rows])
    return x, y


def gaussian_weights(x, x0, tau):
    return np.exp(-((x - x0) ** 2) / (2 * tau ** 2))


def local_linear_fit(x, y, x0, tau):
    """Fit a weighted local line at query point x0 and return the
    prediction y_hat(x0), using the weighted analogue of the closed-form
    least-squares solution."""
    weights = gaussian_weights(x, x0, tau)
    w_sum = np.sum(weights)

    x_mean_w = np.sum(weights * x) / w_sum
    y_mean_w = np.sum(weights * y) / w_sum

    numerator = np.sum(weights * (x - x_mean_w) * (y - y_mean_w))
    denominator = np.sum(weights * (x - x_mean_w) ** 2)

    w_coef = numerator / denominator
    b_coef = y_mean_w - w_coef * x_mean_w

    return w_coef * x0 + b_coef


def locally_weighted_regression(x, y, query_points, tau):
    return np.array([local_linear_fit(x, y, x0, tau) for x0 in query_points])


def main():
    x, y = load_csv(CSV_PATH)
    print(f"Loaded {len(x)} points from {CSV_PATH}")

    query_points = np.linspace(x.min(), x.max(), N_QUERY_POINTS)
    y_pred = locally_weighted_regression(x, y, query_points, TAU)

    # Residuals evaluated at the training points themselves, for a
    # rough overall fit quality measure.
    y_fit_at_data = locally_weighted_regression(x, y, x, TAU)
    mse = np.mean((y_fit_at_data - y) ** 2)
    print(f"MSE at training points (tau={TAU}): {mse:.6f}")

    plt.figure(figsize=(6.5, 4.5))
    plt.scatter(x, y, alpha=0.6, label="data")
    plt.plot(query_points, y_pred, color="red", linewidth=2,
              label=f"locally weighted fit (tau={TAU})")
    plt.xlabel(X_COL)
    plt.ylabel(Y_COL)
    plt.title("Locally weighted regression")
    plt.legend()
    plt.tight_layout()
    plt.savefig("locally_weighted_regression_fit.png", dpi=150)
    print("Saved plot to locally_weighted_regression_fit.png")


if __name__ == "__main__":
    main()

## Maximum Likelihood Estimation

In [ ]:
"""
Demo of Maximum Likelihood Estimation (MLE) for the parameter p of a
Bernoulli distribution, given a sequence of coin flips (0 = tails,
1 = heads).

Model:  P(y_i = 1) = p,  P(y_i = 0) = 1 - p,  for i = 1, ..., n

Likelihood of the observed data (n flips, k heads):
    L(p) = p^k * (1-p)^(n-k)

Log-likelihood:
    log L(p) = k*log(p) + (n-k)*log(1-p)

Setting d(log L)/dp = 0 gives the closed-form MLE:
    p_hat = k / n

This script generates synthetic coin-flip data with a known true p,
computes the closed-form MLE, and plots the log-likelihood curve over
a range of candidate p values with the MLE marked, to show visually
that the closed-form solution is indeed the peak of the curve.
"""

import numpy as np
import matplotlib.pyplot as plt

TRUE_P = 0.7   # true (unknown, in practice) probability of heads
N_FLIPS = 50
SEED = 0


def log_likelihood(p, k, n):
    """Log-likelihood of observing k heads out of n flips, as a
    function of candidate parameter p."""
    return k * np.log(p) + (n - k) * np.log(1 - p)


def main():
    rng = np.random.default_rng(SEED)
    flips = rng.binomial(1, TRUE_P, N_FLIPS)
    k = flips.sum()
    n = len(flips)

    # Closed-form MLE.
    p_hat = k / n
    print(f"Flips: {n}, heads: {k}")
    print(f"Closed-form MLE: p_hat = {p_hat:.4f}")
    print(f"True p used to generate the data: {TRUE_P}")

    # Evaluate the log-likelihood over a grid of candidate p values,
    # avoiding the endpoints 0 and 1 where log(0) is undefined.
    p_grid = np.linspace(0.001, 0.999, 500)
    log_lik = log_likelihood(p_grid, k, n)

    plt.figure(figsize=(7, 4.5))
    plt.plot(p_grid, log_lik, label="log-likelihood, log L(p)")
    plt.axvline(p_hat, color="red", linestyle="--",
                label=f"MLE: p_hat = {p_hat:.3f}")
    plt.xlabel("candidate parameter p")
    plt.ylabel("log-likelihood")
    plt.title(f"Log-likelihood of {k} heads in {n} flips")
    plt.legend()
    plt.tight_layout()
    plt.savefig("mle_bernoulli.png", dpi=150)
    print("Saved plot to mle_bernoulli.png")


if __name__ == "__main__":
    main()